# ArcGIS Online item description editor

**Welcome!**  
This notebook helps you scan, review, and update ArcGIS Online item Terms of Use HTML (the `licenseInfo` field) at scale.

**Where this notebook can run**  
- VS Code on macOS (local Jupyter kernel).
- VS Code on Windows (local Jupyter kernel).
- ArcGIS Online Notebook (JupyterLab-style).

**How to use this notebook**  
- Start with **1. Setup and authenticate** and run the code cell to connect to your organization.
- Run **2. Scan your content** to find matching `licenseInfo` terms.
- Save scan outputs, optionally run a secondary scan for additional terms, and review strict matches.
- Run a **dry run** to preview exactly what would change.
- Generate the side-by-side report to visually compare old and new HTML.
- Commit updates only after validating the dry-run/report outputs.

**Notes**  
- Some org-wide scans take time; progress messages are printed while users and folders are traversed.
- The workflow is designed to be safe-by-default: review first, then update.
- If you need to restart, use Kernel restart and rerun from setup.

**tldr;**

In [1]:
# Run this cell to display Notebook details
from IPython.display import display, Markdown

# Display details of what this notebook does
tldr_md = """
**What this notebook does**  
- Authenticates to ArcGIS Online (multiple contexts)
- Scans an entire Organization's Item Description page's "Terms of Use" section (aka `licenseInfo`).
- Finds items that match one or more target strings (for example, outdated logo URLs or legacy text snippets).
- Exports scan outputs for auditability (default filenames:`scan_matches.csv` and `scan_errors.csv`).
- Supports optional secondary scans to target additional terms while excluding already matched item IDs. (improves scan speed and accuracy)
- Builds a dry-run update plan that shows old vs new HTML before any edit is applied.
- Generates a user friendly side-by-side HTML review report for visual QA.
- Applies updates and edits the itemsonly after explicit confirmation, then exports success/error results.
- Works in local VS Code notebooks (macOS and Windows) and ArcGIS Online notebooks.
"""
display(Markdown(tldr_md))


**What this notebook does**  
- Authenticates to ArcGIS Online (multiple contexts)
- Scans an entire Organization's Item Description page's "Terms of Use" section (aka `licenseInfo`).
- Finds items that match one or more target strings (for example, outdated logo URLs or legacy text snippets).
- Exports scan outputs for auditability (default filenames:`scan_matches.csv` and `scan_errors.csv`).
- Supports optional secondary scans to target additional terms while excluding already matched item IDs. (improves scan speed and accuracy)
- Builds a dry-run update plan that shows old vs new HTML before any edit is applied.
- Generates a user friendly side-by-side HTML review report for visual QA.
- Applies updates and edits the itemsonly after explicit confirmation, then exports success/error results.
- Works in local VS Code notebooks (macOS and Windows) and ArcGIS Online notebooks.


## 1. Setup and authenticate

In [ ]:
# In the toolbar above, select View > Collapse All Code to hide the code.
print("Initializing...")

# Cell 1. Import packages, configure paths, authenticate, and load helper functions
import sys
from pathlib import Path
import pandas as pd
import ipywidgets as widgets

# Support local VS Code (macOS/Windows) and ArcGIS Online Notebook locations for helper_functions.py
NOTEBOOK_DIR = Path.cwd()
CANDIDATE_HELPER_DIRS = [NOTEBOOK_DIR, Path("/arcgis/home")]
for p in CANDIDATE_HELPER_DIRS:
    helper_file = p / "helper_functions.py"
    if helper_file.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

from helper_functions import (
    detect_environment,
    authenticate_gis,
    initialize_ui,
    set_runtime_context,
    setup_notebook_btn,
    parse_target_terms,
    run_primary_scan_btn,
    save_scan_outputs_btn,
    run_secondary_scan_btn,
    run_strict_match_filter_btn,
    dry_run_btn,
    create_report_btn,
    export_dry_run_btn,
    apply_updates_btn,
    export_final_results_btn,
    OFFICIAL_TOU_HTML_FILE,
 )

# Set Pandas dataframe display options
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", 1000)

# Define shared notebook state
context = {
    "gis": None,
    "matches_df": None,
    "errors_df": None,
    "all_items_df": None,
    "TARGET_STRINGS": [],
    "official_tou_html_file": OFFICIAL_TOU_HTML_FILE,
}
set_runtime_context(context)

# Detect the current environment
current_env, env_string = detect_environment()
print(f"Currently running in {env_string}")

output1 = initialize_ui("output")
context["output1"] = output1

# Create widgets
btn1 = initialize_ui("button", description="Setup Notebook", width="200px")
display(btn1)
btn1.on_click(setup_notebook_btn)
display(output1)

Initializing...
Currently running in VSCode Notebook environment


Button(description='Setup Notebook', layout=Layout(height='40px', width='200px'), style=ButtonStyle())

Output()

## 2. Scan your content
This cell scans your content for the text strings you input into the "Search strings:" box.

In [ ]:
# Cell 2: Define terms and scan org content for licenseInfo matches
output2 = initialize_ui("output")
context["output2"] = output2

txt2 = initialize_ui("label", value="Enter text/URLs to find in the Terms of Use section -->", width="350px")
txt2_2 = initialize_ui("label", value="Separate multiple terms with commas: ['term1', 'term2']", width="350px")
txt2_3 = initialize_ui("label", value="And enclose any spaces in single quotes: ['\"term with spaces\"', 'another term']", width="350px")
input2 = initialize_ui(
    "textarea",
    value='["https://downloads.esri.com/blogs/arcgisonline/esrilogo_new.png", "esrilogo"]',
    description="",
    width="500px",
    height="70px",
)
context["input2"] = input2
vbox2 = widgets.VBox([txt2, txt2_2, txt2_3])
hbox2 = initialize_ui("hbox", elements=[vbox2, input2])
btn2 = initialize_ui("button", description="Scan for items", width="200px")
display(widgets.VBox([hbox2, btn2, output2]))

btn2.on_click(run_primary_scan_btn)

## 3. Save the scan results to csv files
This cell saves the scan. You can change the filenames and location if you'd like

In [ ]:
# Cell 3: Save latest scan outputs to CSV
output3 = initialize_ui("output")
context["output3"] = output3
btn3 = initialize_ui("button", description="Save files")
display(widgets.VBox([btn3, output3]))

btn3.on_click(save_scan_outputs_btn)

## 4. Optionally reload results from a previous scan
use csv

In [ ]:
context["matches_df"] = pd.read_csv("scan_matches.csv", dtype={"item_id": str})

try:
    context["errors_df"] = pd.read_csv("scan_errors.csv")
except pd.errors.EmptyDataError:
    context["errors_df"] = pd.DataFrame(columns=["username", "error"])

context["all_items_df"] = pd.read_csv("scan_all_items.csv", dtype={"item_id": str})

print(
    f"Reloaded: matches={len(context['matches_df'])}, "
    f"errors={len(context['errors_df'])}, "
    f"all_items={len(context['all_items_df'])}"
)

## 5. Secondary scan
This cell saves you time if you want to search additional terms.
If you want to search again, click the SECONDARY_SCAN check box and add the new terms to NEW_TERMS and run the cell below

In [ ]:
# Cell 5: Optional secondary scan using new terms and excluding previous matches
output4 = initialize_ui("output")
context["output4"] = output4
checkbox4 = initialize_ui("checkbox", description="Run secondary scan with new search terms?", value=False)
context["checkbox4"] = checkbox4
input4 = initialize_ui(
    "textarea",
    value='["https://www.esri.com/content/dam/esrisites/en-us/common/logos/esri-logo.jpg"]',
    description="New search strings:",
    width="500px",
    height="70px",
)
context["input4"] = input4

btn4 = initialize_ui("button", description="Run secondary scan")
display(widgets.VBox([checkbox4, input4, btn4, output4]))

btn4.on_click(run_secondary_scan_btn)

## 6. Strict matching output

In [ ]:
# Cell 6: Optionally filter matches_df to rows containing one exact term
output5 = initialize_ui("output")
context["output5"] = output5
exact_term_input = initialize_ui(
    "text",
    value="https://downloads.esri.com/blogs/arcgisonline/esrilogo_new.png",
    description="Exact term:",
    width="700px",
)
context["exact_term_input"] = exact_term_input
btn5 = initialize_ui("button", description="Filter exact matches", width="200px")
display(widgets.VBox([exact_term_input, btn5, output5]))

btn5.on_click(run_strict_match_filter_btn)

## 7. Dry run

In [ ]:
# Cell 7: Build a dry-run plan before making any changes
output6 = initialize_ui("output")
context["output6"] = output6
tou_file_path_input = initialize_ui(
    "text",
    value=context.get("official_tou_html_file", OFFICIAL_TOU_HTML_FILE),
    description="ToU HTML file:",
    width="900px",
)
btn6 = initialize_ui("button", description="Build dry run", width="180px")

def run_dry_run_with_file(_button):
    entered = (tou_file_path_input.value or "").strip()
    context["official_tou_html_file"] = entered or OFFICIAL_TOU_HTML_FILE
    dry_run_btn(_button)

display(widgets.VBox([tou_file_path_input, btn6, output6]))

btn6.on_click(run_dry_run_with_file)

## 8. Export dry run results

In [ ]:
# Cell 8: Export the dry-run plan for record-keeping and review
output7 = initialize_ui("output")
context["output7"] = output7
btn7 = initialize_ui("button", description="Export dry-run CSV", width="200px")
display(widgets.VBox([btn7, output7]))

btn7.on_click(export_dry_run_btn)

## 9. Create report

In [ ]:
# Cell 9: Generate an HTML report for side-by-side review before updating items
output8 = initialize_ui("output")
context["output8"] = output8
btn8 = initialize_ui("button", description="Create review report", width="200px")
display(widgets.VBox([btn8, output8]))

btn8.on_click(create_report_btn)

## 10. Commit updates

In [ ]:
# Cell 10: Apply updates only after reviewing the dry run/report and optional selected IDs
output9 = initialize_ui("output")
context["output9"] = output9
selected_ids_path = initialize_ui(
    "text",
    value="selected_item_ids.json",
    description="Selected IDs file:",
    width="700px",
)
context["selected_ids_path"] = selected_ids_path
btn9 = initialize_ui("button", description="Apply updates", width="180px")
display(widgets.VBox([selected_ids_path, btn9, output9]))

btn9.on_click(apply_updates_btn)

## 11. Export final results

In [ ]:
# Cell 11: Export final update results to CSV files for record-keeping
output10 = initialize_ui("output")
context["output10"] = output10
btn10 = initialize_ui("button", description="Export final CSVs", width="180px")
display(widgets.VBox([btn10, output10]))

btn10.on_click(export_final_results_btn)